# Week 1 — Networks

Brief: [briefs/week1.md](briefs/week1.md)

Scratch cells are fine here. The post is written from the **Findings** cell at the bottom, and every number that ends up in the post must be traceable to a cell in this notebook.

In [ ]:
import sys
from pathlib import Path

# Make the repo's helper package importable whether the kernel started in notebooks/ or the repo root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "marvel").is_dir())
sys.path.insert(0, str(ROOT))

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from marvel import load_graph, load_nodes, save_figure, apply_style

apply_style()
WEEK = 1

In [ ]:
G = load_graph()  # roster first, then edges: the 17 isolates survive
assert G.number_of_nodes() == 303 and G.number_of_edges() == 1784
print(G)
print("isolates:", sorted(nx.isolates(G)))

## What we asked

_Write the question here before touching the data._

## What we did

_Method notes as you go: which measures, which choices, and why._

In [ ]:
# One table per character: display name, in-degree, out-degree.
deg = pd.DataFrame({
    "name": [G.nodes[n]["name"] for n in G],
    "in": pd.Series(dict(G.in_degree())),
    "out": pd.Series(dict(G.out_degree())),
})

print("nodes:", len(deg))
print("in-degree 0:", (deg["in"] == 0).sum(),
      "| out-degree 0:", (deg["out"] == 0).sum(),
      "| both (isolates):", ((deg["in"] == 0) & (deg["out"] == 0)).sum())

In [ ]:
# The two top-ten lists the post is about.
top_in = deg.nlargest(10, "in")[["name", "in", "out"]]
top_out = deg.nlargest(10, "out")[["name", "in", "out"]]

print("TOP 10 BY IN-DEGREE (linked to most)")
print(top_in.to_string(index=False))
print("\nTOP 10 BY OUT-DEGREE (link out most)")
print(top_out.to_string(index=False))
print("\nin both lists:", sorted(set(top_in["name"]) & set(top_out["name"])))

In [ ]:
# in/out as a candidate fame index. Two things to decide before reading it:
#   - out-degree 0 makes the ratio undefined, so those characters drop out;
#   - the ratio ignores size, so a 1-in/1-out character scores like a 50/50 one.
ratio = deg[deg["out"] > 0].assign(ratio=lambda d: d["in"] / d["out"])
print("scored:", len(ratio), "of", len(deg), "(dropped:", (deg["out"] == 0).sum(), "with out-degree 0)")

print("\nHIGHEST in/out")
print(ratio.nlargest(10, "ratio").round(2).to_string(index=False))
print("\nLOWEST in/out")
print(ratio.nsmallest(10, "ratio").round(2).to_string(index=False))

# Same index restricted to the ten best-connected characters, where the ties vanish.
print("\nin/out among the ten highest total-degree characters")
print(deg.assign(total=deg["in"] + deg["out"])
         .nlargest(10, "total")
         .assign(ratio=lambda d: (d["in"] / d["out"]).round(2))[["name", "in", "out", "ratio"]]
         .to_string(index=False))

In [ ]:
# Paste-ready GFM for the post. df.to_markdown() would do this, but it needs
# tabulate, and the notebook has to run on a clean checkout at publish time.
def to_gfm(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


print("TOP 10 BY IN-DEGREE\n" + to_gfm(top_in))
print("\nTOP 10 BY OUT-DEGREE\n" + to_gfm(top_out))

## Findings

Fill this in as you go. The post is written from here and from nothing else.

- **What we asked:**

Do famous characters mainly have Wikipedia articles about themselves, while obscure characters mainly tell their story through their relationships to better-known characters? If so, we should see well-known characters with a higher in-degree than out-degree, and the reverse for obscure ones.

- **What we did:**

We took the ten articles with the highest in-degree and the ten with the highest out-degree, and made a subjective check of whether we had heard of each character. In addition, we calculated the ratio between in- and out-degree to see whether it could act as an indicator of fame and obscurity.

- **The figure or table:** paste the include line printed by `save_figure()`

Top 10 by in-degree (linked to most):

| name | in | out |
|---|---|---|
| Spider-Man | 106 | 9 |
| Hulk | 64 | 10 |
| Wolverine (character) | 60 | 17 |
| Doctor Strange | 50 | 17 |
| Deadpool | 33 | 19 |
| She-Hulk | 29 | 20 |
| Scarlet Witch | 28 | 14 |
| Black Panther (character) | 27 | 11 |
| Cyclops (Marvel Comics) | 26 | 11 |
| Luke Cage | 25 | 9 |

Top 10 by out-degree (link out most):

| name | in | out |
|---|---|---|
| Betsy Braddock | 7 | 28 |
| Cloak and Dagger (characters) | 11 | 24 |
| Adam Warlock | 8 | 22 |
| Venom (character) | 17 | 21 |
| She-Hulk | 29 | 20 |
| U.S. Agent | 6 | 20 |
| Deadpool | 33 | 19 |
| Rachel Summers | 16 | 19 |
| Noh-Varr | 4 | 18 |
| Doctor Strange | 50 | 17 |

- **What surprised us:**

Spider-Man and Betsy Braddock each fit one side of our hypothesis, but the inclusion of the well-known characters Venom, Doctor Strange and Deadpool breaks it.